# 07 — Supervised Learning: Classification

**Topics:** LogisticRegression, DecisionTree, RandomForest, SVM, evaluation metrics (precision, recall, F1, ROC-AUC, confusion matrix).

**Reference:** [sklearn classification](https://scikit-learn.org/stable/supervised_learning.html)

**Dataset:** Credit card fraud detection — heavily imbalanced binary classification.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve
)

# Credit-g: German credit data, binary classification (good/bad credit)
credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = pd.get_dummies(credit_raw, drop_first=True)
X = credit.drop('class_good', axis=1)
y = credit['class_good'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Class distribution: {y.value_counts().to_dict()}")
print(X.shape)

---
## Exercise 1 — Classification Report Function

Build a reusable `evaluate_classifier(model, X_test, y_test, model_name)` function that returns a dict with:
`accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `confusion_matrix` (as a list of lists).

All scalar metrics rounded to 4dp. Use `average='binary'` for precision/recall/f1.

In [ ]:
def evaluate_classifier(model, X_test, y_test, model_name: str = '') -> dict:
    """
    Returns full classification evaluation dict.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Quick test — we'll use this function throughout
lr = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
lr.fit(X_train, y_train)
lr_eval = evaluate_classifier(lr, X_test, y_test, 'LogisticRegression')

assert set(lr_eval.keys()) == {'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'confusion_matrix'}
assert 0 < lr_eval['roc_auc'] <= 1
assert len(lr_eval['confusion_matrix']) == 2
print("✓ Exercise 1 passed")
print(lr_eval)

---
## Exercise 2 — Decision Tree & Overfitting

1. Train `DecisionTreeClassifier` at `max_depth` values `[1, 2, 3, 5, 8, 12, None]`.
2. For each depth, record `train_accuracy` and `test_accuracy`.
3. Return `depth_results`: DataFrame with columns `max_depth`, `train_accuracy`, `test_accuracy`, `overfit_gap` (train - test).
4. Identify the `best_depth`: depth with highest test accuracy.
5. In a markdown cell, explain what you observe about train vs test accuracy as depth increases.

In [ ]:
def analyze_tree_depth(X_train, y_train, X_test, y_test):
    """
    Returns (depth_results DataFrame, best_depth)
    """
    # YOUR CODE HERE
    pass

depth_results, best_depth = analyze_tree_depth(X_train, y_train, X_test, y_test)

In [ ]:
# --- ASSERTIONS ---
assert list(depth_results.columns) == ['max_depth', 'train_accuracy', 'test_accuracy', 'overfit_gap']
assert len(depth_results) == 7
assert depth_results.loc[depth_results['max_depth'].isna(), 'overfit_gap'].values[0] > 0.05, "Full tree should overfit"
print(f"✓ Exercise 2 passed — Best depth: {best_depth}")
print(depth_results)

---
## Exercise 3 — Random Forest & Feature Importance

1. Train `RandomForestClassifier(n_estimators=100, random_state=42)`.
2. Extract `feature_importances_` and build `rf_importance_df`: columns `feature`, `importance`, sorted descending.
3. Compute **permutation importance** manually for the top 5 features:
   - For each feature in top 5: shuffle that feature column in `X_test`, predict, record accuracy drop.
   - Return `perm_importance_df`: columns `feature`, `accuracy_drop`.
4. Compare sklearn importance vs permutation importance — are they consistent?

In [ ]:
def analyze_rf_importance(X_train, y_train, X_test, y_test, feature_names):
    """
    Returns (rf_model, rf_importance_df, perm_importance_df)
    """
    # YOUR CODE HERE
    pass

rf_model, rf_importance_df, perm_importance_df = analyze_rf_importance(X_train, y_train, X_test, y_test, X.columns.tolist())

In [ ]:
# --- ASSERTIONS ---
assert list(rf_importance_df.columns) == ['feature', 'importance']
assert rf_importance_df['importance'].is_monotonic_decreasing
assert abs(rf_importance_df['importance'].sum() - 1.0) < 1e-6, "Importances must sum to 1"
assert list(perm_importance_df.columns) == ['feature', 'accuracy_drop']
assert len(perm_importance_df) == 5
print("✓ Exercise 3 passed")
print(rf_importance_df.head())

---
## Exercise 4 — Threshold Tuning & Precision-Recall Tradeoff

**Context:** In credit risk, false negatives (approving bad loans) are more costly than false positives. Adjust decision threshold accordingly.

Using the `LogisticRegression` pipeline from Exercise 1:

1. Get predicted probabilities for class 1.
2. Evaluate the model at thresholds `np.arange(0.1, 0.9, 0.05)`.
3. For each threshold, compute: `precision`, `recall`, `f1`, `n_predicted_positive`.
4. Return `threshold_df`: DataFrame with these 5 columns (including `threshold`).
5. Find `optimal_threshold_f1`: the threshold maximizing F1.
6. Find `recall_90_threshold`: the lowest threshold achieving recall ≥ 0.90.

In [ ]:
def threshold_analysis(model, X_test, y_test):
    """
    Returns (threshold_df, optimal_threshold_f1, recall_90_threshold)
    """
    # YOUR CODE HERE
    pass

threshold_df, optimal_threshold_f1, recall_90_threshold = threshold_analysis(lr, X_test, y_test)

In [ ]:
# --- ASSERTIONS ---
assert list(threshold_df.columns) == ['threshold', 'precision', 'recall', 'f1', 'n_predicted_positive']
assert 0 < optimal_threshold_f1 < 1
assert recall_90_threshold < optimal_threshold_f1, "To achieve 90% recall, threshold should be lower"

# At recall_90_threshold, recall should be >= 0.90
row = threshold_df[threshold_df['threshold'] == recall_90_threshold]
assert row['recall'].values[0] >= 0.90

print(f"✓ Exercise 4 passed")
print(f"Optimal F1 threshold: {optimal_threshold_f1:.2f} | Recall 90% threshold: {recall_90_threshold:.2f}")

---
## Exercise 5 — ROC Curve & AUC Comparison

1. Train all 4 classifiers: LogisticRegression, DecisionTree (best_depth), RandomForest(100), SVC(probability=True).
2. For each, compute the full ROC curve (fpr, tpr arrays) and AUC.
3. Return `roc_results`: dict mapping model name → `{fpr, tpr, auc}`.
4. Return `auc_comparison`: DataFrame with `model` and `auc`, sorted descending.

In [ ]:
def compare_roc_curves(X_train, y_train, X_test, y_test, best_depth):
    """
    Returns (roc_results dict, auc_comparison DataFrame)
    """
    # YOUR CODE HERE
    pass

roc_results, auc_comparison = compare_roc_curves(X_train, y_train, X_test, y_test, best_depth)

In [ ]:
# --- ASSERTIONS ---
assert len(roc_results) == 4
for model_name, result in roc_results.items():
    assert 'fpr' in result and 'tpr' in result and 'auc' in result
    assert 0.5 < result['auc'] <= 1.0, f"{model_name} AUC should be > 0.5"
assert auc_comparison['auc'].is_monotonic_decreasing
print("✓ Exercise 5 passed")
print(auc_comparison)

---
## Exercise 6 — Handling Class Imbalance

**Context:** The dataset has ~70% good / 30% bad loans. Explore strategies for imbalanced classification.

1. Train `LogisticRegression` with `class_weight='balanced'` vs default. Compare F1 and recall for the minority class.
2. Train `RandomForestClassifier` with `class_weight='balanced'` vs default.
3. Implement manual **oversampling** of the minority class (no imbalanced-learn): randomly sample with replacement until classes are equal. Call it `X_train_over`, `y_train_over`.
4. Train LogisticRegression on oversampled data.
5. Return `imbalance_results`: DataFrame comparing all 5 variants on `f1`, `recall`, `precision` for the minority class.

In [ ]:
def handle_imbalance(X_train, y_train, X_test, y_test):
    """
    Returns imbalance_results DataFrame.
    """
    # YOUR CODE HERE
    pass

imbalance_results = handle_imbalance(X_train, y_train, X_test, y_test)

In [ ]:
# --- ASSERTIONS ---
assert len(imbalance_results) == 5
for col in ['f1', 'recall', 'precision']:
    assert col in imbalance_results.columns
print("✓ Exercise 6 passed")
print(imbalance_results)

---
## Exercise 7 — Full Classifier Benchmark

Build the final comparison table `classifier_benchmark`: all 4 classifiers evaluated on all metrics:
`accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `cv_f1_mean` (5-fold stratified CV).

Sort by `roc_auc` descending. Index = model name. This is the kind of table you'd present in a technical interview.

In [ ]:
def build_classifier_benchmark(X_train, y_train, X_test, y_test):
    """
    Returns classifier_benchmark DataFrame.
    """
    # YOUR CODE HERE
    pass

classifier_benchmark = build_classifier_benchmark(X_train, y_train, X_test, y_test)

In [ ]:
# --- ASSERTIONS ---
expected_cols = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'cv_f1_mean']
for col in expected_cols:
    assert col in classifier_benchmark.columns
assert len(classifier_benchmark) == 4
assert classifier_benchmark['roc_auc'].is_monotonic_decreasing
print("✓ Exercise 7 passed")
print(classifier_benchmark)